# Wind-based indices vs $W_{MLD}$: RMSE table (tab:wind_performance)

All in m$^2$ s$^{-1}$ per metre of coastline, 5–16°S, reference $W_{MLD}$ (`W_L` of `CUTI_BEUTI_HR_L_C.nc`, as in UI_comparison).

- **Seasonality**: RMSE between the monthly climatologies; [bias] = mean(index − $W_{MLD}$).
- **Non-extreme**: detrended 13-month smoothed anomalies, extreme El Niño months excluded.
- **Extreme El Niño**: the same anomalies, El Niño months only.

In [1]:
import numpy as np
import pandas as pd
import xarray as xr

DATA = '../data/'
lat1, lat0 = -16, -5
NINO = [(pd.Timestamp('1982-06-01'), pd.Timestamp('1983-12-31')),
        (pd.Timestamp('1997-04-01'), pd.Timestamp('1998-07-31'))]

### Data

In [2]:
# CUTI and W_MLD per metre of coastline (file stored N -> S)
hr = xr.open_dataset(DATA + 'CUTI_BEUTI_HR_L_C.nc').sortby('lat_band').sel(lat_band=slice(lat1, lat0)).compute()
WMLD_L = hr.W_L.mean('lat_band').rename('W_MLD')
CUTI   = hr.CUTI.mean('lat_band').rename('CUTI')
U_ek   = hr.U_ek.mean('lat_band').rename('U_ek')   # Ekman part of CUTI

# AV (CUTI band, from AV.ipynb) -> per metre of coastline: sum(AV dx) * dy / L_coast
R  = 6371000.0
L  = hr.L_coast.rename(lat_band='lat')
dy = hr.dy.rename(lat_band='lat')
AV = xr.open_dataset(DATA + 'AV.nc').AV.sortby('lat').sel(lat=slice(lat1, lat0)).compute()
dx = R * np.cos(np.deg2rad(AV.lat)) * np.deg2rad(1 / 12)
AV_L = ((AV * dx).sum('lon', min_count=1)
        * dy.reindex(lat=AV.lat, method='nearest', tolerance=0.02)
        / L.reindex(lat=AV.lat, method='nearest', tolerance=0.02)).mean('lat').rename('AV')

# Ek_t: cross-shore Ekman transport at the band edge (offshore=0), m2/s per metre of coastline
Ek_t = xr.open_dataset(DATA + 'Winds_components_Lu.nc').U_ek_cross.isel(offshore=0)
Ek_t = (Ek_t.swap_dims({'coast_point': 'lat'})
            .drop_vars(['coast_point', 'lon', 'bearing', 'distance', 'f', 'offshore'])
            .sortby('lat').sel(lat=slice(lat1, lat0)).mean('lat').rename('Ek_t').compute())

### Helpers (same processing as UI_comparison: anomalies → detrend → 13-month centred mean)

In [3]:
def to_series(da):
    """Put the time axis on month starts so all series line up."""
    da = da.assign_coords(time=pd.to_datetime(da.time.dt.strftime('%Y-%m-01').values))
    return da.drop_vars([c for c in da.coords if c != 'time'])

def anomalies(da):
    return (da.groupby('time.month') - da.groupby('time.month').mean('time')).drop_vars('month', errors='ignore')

def detrend(da):
    t = np.arange(da.sizes['time'], dtype=float)
    b, a = np.polyfit(t, da.values.astype(float), 1)
    return da - (a + b * t)

def prep_ia(da, window=13):
    return detrend(anomalies(to_series(da))).rolling(time=window, center=True).mean()

def en_mask(time):
    t = pd.DatetimeIndex(time.values)
    m = np.zeros(t.size, bool)
    for s, e in NINO:
        m |= (t >= s) & (t <= e)
    return m

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

### Table

In [4]:
ref    = to_series(WMLD_L)
ref_ia = prep_ia(WMLD_L)
clim_ref = ref.groupby('time.month').mean('time')

rows = []
for name, da in {'Ek_t': Ek_t, 'AV': AV_L, 'U_ek': U_ek, 'CUTI': CUTI}.items():
    # seasonality
    clim = to_series(da).groupby('time.month').mean('time')
    d = (clim - clim_ref).values
    # interannual: smoothed detrended anomalies on the common months
    x, y = xr.align(ref_ia, prep_ia(da), join='inner')
    ok = np.isfinite(x.values) & np.isfinite(y.values)
    en = en_mask(x.time)
    rows.append(dict(index=name,
                     rmse_seas=float(np.sqrt(np.mean(d ** 2))), bias_seas=float(d.mean()),
                     rmse_nonEN=rmse(y.values[ok & ~en], x.values[ok & ~en]),
                     rmse_EN=rmse(y.values[ok & en], x.values[ok & en]),
                     N_nonEN=int((ok & ~en).sum()), N_EN=int((ok & en).sum())))

wind_perf = pd.DataFrame(rows).set_index('index')
print(wind_perf.round(3))

       rmse_seas  bias_seas  rmse_nonEN  rmse_EN  N_nonEN  N_EN
index                                                          
Ek_t       0.565      0.562       0.109    0.383      385    35
AV         0.209     -0.104       0.105    0.266      385    35
CUTI       0.282      0.243       0.089    0.121      385    35


In [5]:
# LaTeX rows, lowest RMSE per column in bold
best = {c: wind_perf[c].idxmin() for c in ['rmse_seas', 'rmse_nonEN', 'rmse_EN']}
def cell(c, k):
    v = wind_perf.loc[k, c]
    return f'\\textbf{{{v:.2f}}}' if best[c] == k else f'${v:.2f}$'

lab = {'Ek_t': '$Ek_t$', 'AV': 'AV    ', 'U_ek': '$U^{Ek}$', 'CUTI': 'CUTI  '}
for k in wind_perf.index:
    print(f"{lab[k]} & {cell('rmse_seas', k)} \\, [${wind_perf.loc[k, 'bias_seas']:+.2f}$]"
          f"& {cell('rmse_nonEN', k)}& {cell('rmse_EN', k)}\\\\")

$Ek_t$ & $0.57$ \, [$+0.56$]& $0.11$& $0.38$\\
AV     & \textbf{0.21} \, [$-0.10$]& $0.10$& $0.27$\\
CUTI   & $0.28$ \, [$+0.24$]& \textbf{0.09}& \textbf{0.12}\\


In [6]:
# Amplitude of the seasonal cycle: sigma of each 12-month climatology (ddof=0, as in UI_comparison)
# and its difference relative to W_MLD; seasonal r for reference
sig_ref = float(clim_ref.std())
print(f"{'index':6s} {'sigma':>7s} {'vs W_MLD':>9s} {'r_seas':>7s}")
print(f"{'W_MLD':6s} {sig_ref:7.2f} {'':>9s} {'':>7s}")
for name, da in {'Ek_t': Ek_t, 'AV': AV_L, 'U_ek': U_ek, 'CUTI': CUTI}.items():
    clim = to_series(da).groupby('time.month').mean('time')
    sig  = float(clim.std())
    r    = float(np.corrcoef(clim.values, clim_ref.values)[0, 1])
    print(f"{name:6s} {sig:7.2f} {100 * (sig / sig_ref - 1):+8.0f}% {r:7.3f}")


index    sigma  vs W_MLD  r_seas
W_MLD     0.50                  
Ek_t      0.52       +4%   0.993
AV        0.33      -34%   0.990
CUTI      0.37      -26%   0.993
